# NeuroGen Analysis

Visualize benchmark results and diagnostic metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import glob

plt.style.use('seaborn-v0_8-whitegrid')
OUTPUTS = Path('outputs')

## Load Latest Benchmark

In [ ]:
csvs = sorted(OUTPUTS.glob('benchmark_*.csv'))
if not csvs:
    raise FileNotFoundError('No benchmark CSVs found. Run: uv run benchmark.py --compare "default,xavier" --seeds 5')
df = pd.read_csv(csvs[-1])
print(f'Loaded {csvs[-1].name}: {len(df)} runs, {df["method"].nunique()} methods')
df.head()

## val_bpb by Init Method

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
grouped = df.groupby('method')['val_bpb']
means = grouped.mean().sort_values()
stds = grouped.std().reindex(means.index)
means.plot(kind='bar', yerr=stds, ax=ax, capsize=4, color='steelblue', edgecolor='black')
ax.set_ylabel('val_bpb (lower is better)')
ax.set_title('Validation BPB by Initialization Method')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUTS / 'val_bpb_comparison.png', dpi=150)
plt.show()

## Init Diagnostic Metrics

In [ ]:
diag_cols = ['init_head_diversity', 'init_block_diag_ratio', 'init_layer_similarity', 'init_weight_std']
available = [c for c in diag_cols if c in df.columns]

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 5))
    if len(available) == 1:
        axes = [axes]
    for ax, col in zip(axes, available):
        grouped = df.groupby('method')[col]
        means = grouped.mean().sort_values()
        stds = grouped.std().reindex(means.index)
        means.plot(kind='bar', yerr=stds, ax=ax, capsize=3, color='coral', edgecolor='black')
        ax.set_title(col.replace('init_', ''))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(OUTPUTS / 'init_diagnostics.png', dpi=150)
    plt.show()
else:
    print('No init diagnostic columns found in data')

## Init Loss vs Final val_bpb

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for method, group in df.groupby('method'):
    ax.scatter(group['init_loss'], group['val_bpb'], label=method, s=60, alpha=0.8)
ax.set_xlabel('init_loss')
ax.set_ylabel('val_bpb')
ax.set_title('Does Better Init Translate to Better Final Model?')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS / 'init_vs_final.png', dpi=150)
plt.show()

## Live CA: Gradient Alignment Over Training

Parse `ca_grad_alignment` from run logs. Place log files in `outputs/` to analyze.

In [ ]:
import re

def parse_ca_alignment(log_path):
    """Extract ca_grad_alignment values from a training log."""
    steps, alignments = [], []
    step_pat = re.compile(r'step\s+(\d+)')
    align_pat = re.compile(r'ca_grad_alignment:\s+([\-\d.]+)')
    current_step = 0
    with open(log_path) as f:
        for line in f:
            m = step_pat.search(line)
            if m:
                current_step = int(m.group(1))
            m = align_pat.search(line)
            if m:
                steps.append(current_step)
                alignments.append(float(m.group(1)))
    return steps, alignments

log_files = sorted(OUTPUTS.glob('*.log')) + sorted(Path('.').glob('run.log'))
if log_files:
    fig, ax = plt.subplots(figsize=(8, 4))
    for lf in log_files:
        steps, aligns = parse_ca_alignment(lf)
        if aligns:
            ax.plot(steps, aligns, label=lf.stem, alpha=0.8)
    if ax.get_lines():
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Training Step')
        ax.set_ylabel('CA-Gradient Alignment')
        ax.set_title('CA Cooperation (+1) vs Competition (-1) with Gradient')
        ax.legend()
        plt.tight_layout()
        plt.savefig(OUTPUTS / 'ca_grad_alignment.png', dpi=150)
        plt.show()
    else:
        print('No ca_grad_alignment data found in logs')
else:
    print('No log files found. Run live CA experiments first.')

## Load All Benchmarks (Historical)

In [ ]:
all_csvs = sorted(OUTPUTS.glob('benchmark_*.csv'))
if len(all_csvs) > 1:
    all_df = pd.concat([pd.read_csv(c).assign(benchmark=c.stem) for c in all_csvs])
    print(f'Loaded {len(all_csvs)} benchmarks, {len(all_df)} total runs')
    display(all_df.groupby(['benchmark', 'method'])['val_bpb'].agg(['mean', 'std']).round(4))
else:
    print('Only one benchmark found. Run more to see historical trends.')

## Convergence Plot: val_bpb vs Training Step

The most informative figure — shows whether CA advantage persists, grows, or vanishes over training.
Load loss curves from benchmark JSON output.

In [ ]:
import json
import numpy as np

# Load the most recent loss curves JSON
curve_files = sorted(OUTPUTS.glob('loss_curves_*.json'))
convergence_files = sorted(OUTPUTS.glob('convergence_*.csv'))

if curve_files:
    with open(curve_files[-1]) as f:
        curves = json.load(f)
    print(f'Loaded {curve_files[-1].name}: {len(curves)} runs')

    # Group by method (keys are like "xavier_s42", "xavier+grid_ca_s42")
    method_curves = {}
    for key, curve in curves.items():
        # Parse method name (everything before _s<seed>)
        parts = key.rsplit('_s', 1)
        method = parts[0]
        if method not in method_curves:
            method_curves[method] = []
        method_curves[method].append(curve)

    # Plot: thin lines for individual seeds, thick for mean
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(method_curves)))

    for (method, seed_curves), color in zip(method_curves.items(), colors):
        # Plot individual seeds as thin lines
        for curve in seed_curves:
            steps = [pt[0] for pt in curve]
            bpbs = [pt[2] for pt in curve]
            ax.plot(steps, bpbs, color=color, alpha=0.25, linewidth=1)

        # Compute and plot mean curve
        # Interpolate all seeds to common step grid
        all_steps = sorted(set(s for curve in seed_curves for s, _, _ in curve))
        if all_steps:
            mean_bpbs = []
            for target_step in all_steps:
                vals = []
                for curve in seed_curves:
                    # Find closest step
                    closest = min(curve, key=lambda pt: abs(pt[0] - target_step))
                    vals.append(closest[2])
                mean_bpbs.append(np.mean(vals))
            ax.plot(all_steps, mean_bpbs, color=color, linewidth=2.5, label=method)

    ax.set_xlabel('Training Step')
    ax.set_ylabel('val_bpb (lower is better)')
    ax.set_title('Convergence: val_bpb Over Training')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUTS / 'convergence_plot.png', dpi=150)
    plt.show()

    # Summary: does the gap persist?
    print('\n--- Gap Analysis ---')
    methods_list = list(method_curves.keys())
    if len(methods_list) >= 2:
        m1, m2 = methods_list[0], methods_list[1]
        for curve_set, name in [(method_curves[m1], m1), (method_curves[m2], m2)]:
            finals = [curve[-1][2] for curve in curve_set if curve]
            inits = [curve[0][2] for curve in curve_set if curve]
            print(f'{name}: init={np.mean(inits):.4f} → final={np.mean(finals):.4f}')

elif convergence_files:
    conv_df = pd.read_csv(convergence_files[-1])
    print(f'Loaded convergence data: {convergence_files[-1].name}')

    fig, ax = plt.subplots(figsize=(10, 6))
    for method, group in conv_df.groupby('method'):
        if 'minutes' in group.columns:
            for minutes, mgroup in group.groupby('minutes'):
                ax.scatter(mgroup['total_steps'], mgroup['val_bpb'], label=f'{method} ({minutes:.0f}m)', s=60, alpha=0.7)
        else:
            ax.scatter(group['total_steps'], group['val_bpb'], label=method, s=60, alpha=0.7)
    ax.set_xlabel('Total Steps')
    ax.set_ylabel('val_bpb')
    ax.set_title('Convergence Across Training Horizons')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUTS / 'convergence_plot.png', dpi=150)
    plt.show()
else:
    print('No loss curve data found. Run: uv run benchmark.py --compare "xavier,xavier+grid_ca" --seeds 3')